在 Jupyter Notebook (.ipynb) 中运行 Bash 命令非常简单，主要有两种方式：使用 ! 前缀（单行）或 %%bash 魔术命令（整块代码）。
针对你的服务器健康检查，你可以直接在单元格（Cell）中输入并运行以下代码块：
1. 单行命令 (使用 !)
在任何命令前加一个感叹号，Jupyter 就会将其发送到系统的终端执行。

In [ ]:
!lscpu # AMD Ryzen Threadripper 2990WX 32-Core Processor

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             43 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      64
  On-line CPU(s) list:       0-63
Vendor ID:                   AuthenticAMD
  Model name:                AMD Ryzen Threadripper 2990WX 32-Core Processor
    CPU family:              23
    Model:                   8
    Thread(s) per core:      2
    Core(s) per socket:      32
    Socket(s):               1
    Stepping:                2
    Frequency boost:         enabled
    CPU max MHz:             3000.0000
    CPU min MHz:             2200.0000
    BogoMIPS:                5987.99
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ht s
                             yscall nx mmxext fxsr_opt pdpe1gb rdtscp lm constan
                             t_tsc rep_good no

Corsair H100i RGB platium cooler

In [1]:
# 检查 CPU 实时频率（确认没有被锁频）
!lscpu | grep "MHz"

# 检查 128GB 内存是否全在
!free -h

# 检查 NVIDIA 显卡状态
!nvidia-smi

# 检查所有 13 块硬盘（12块10T + 1块8T SSD）
!lsblk

CPU max MHz:                             3000.0000
CPU min MHz:                             2200.0000
               total        used        free      shared  buff/cache   available
Mem:           125Gi       4.9Gi       115Gi       9.0Mi       5.6Gi       119Gi
Swap:          2.0Gi          0B       2.0Gi
Thu May 28 19:24:57 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.309.01             Driver Version: 535.309.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce GTX 106

2. 块命令 (使用 %%bash)
如果你想一次性运行一堆命令，可以在单元格的第一行写上 %%bash。

In [ ]:
%%bash
echo "=== CPU 核心频率状态 ==="
grep "MHz" /proc/cpuinfo | head -n 5
echo ""
echo "=== 内存状态 ==="
free -h
echo ""
echo "=== 硬盘挂载情况 ==="
df -h | grep "/dev/sd"   # only sd
/dev/nvme0n1p2  916G  130G  739G  15% /
echo ""
echo "=== 温度监控 ==="
sensors | grep -E "Tdie|Tctl|CPU Temperature|GPU"

=== CPU 核心频率状态 ===
cpu MHz		: 2200.000
cpu MHz		: 2200.000
cpu MHz		: 2195.370
cpu MHz		: 1989.302
cpu MHz		: 2198.506

=== 内存状态 ===
               total        used        free      shared  buff/cache   available
Mem:           125Gi       5.0Gi       115Gi       9.0Mi       5.7Gi       119Gi
Swap:          2.0Gi          0B       2.0Gi

=== 硬盘挂载情况 ===
/dev/sda1       7.3T  4.1G  7.3T   1% /mnt/ex_8T_SSD
/dev/sdc1       9.1T   16K  8.6T   1% /srv/external_data
/dev/sdb1       9.1T  7.0T  1.7T  81% /Work_bio

=== 温度监控 ===
Tctl:         +70.9°C  
Tdie:         +43.9°C  
Tctl:         +65.6°C  
Tdie:         +38.6°C  
CPU Temperature:               +41.0°C  
Tctl:         +71.6°C  
Tdie:         +44.6°C  
Tctl:         +68.2°C  
Tdie:         +41.2°C  


3. 整理后的“服务器满血状态检查表” (建议直接复制到 Cell)
你可以把下面这段代码放在一个单元格里运行，它能帮你快速确认昨天的所有修理成果：

In [3]:
print("--- 1. CPU 性能检查 ---")
!lscpu | grep -E "Model name|MHz|CPU\(s\):"

print("\n--- 2. 128GB 内存完整性检查 ---")
!free -h

print("\n--- 3. 120TB+ 磁盘阵列检查 ---")
# 统计有多少块硬盘在线，你应该看到 13 块左右（12个10T机箱+1个8T SSD）
!lsblk | grep "disk"

print("\n--- 4. 显卡状态 ---")
!nvidia-smi --query-gpu=name,temperature.gpu,utilization.gpu,memory.total --format=csv

print("\n--- 5. 核心温度检查 ---")
# 确认 32 核是否处于 40-50 度的健康范围
!sensors | grep "Tdie"

--- 1. CPU 性能检查 ---
CPU(s):                                  64
Model name:                              AMD Ryzen Threadripper 2990WX 32-Core Processor
CPU max MHz:                             3000.0000
CPU min MHz:                             2200.0000
NUMA node0 CPU(s):                       0-7,32-39
NUMA node1 CPU(s):                       16-23,48-55
NUMA node2 CPU(s):                       8-15,40-47
NUMA node3 CPU(s):                       24-31,56-63

--- 2. 128GB 内存完整性检查 ---
               total        used        free      shared  buff/cache   available
Mem:           125Gi       4.6Gi       115Gi       9.0Mi       5.7Gi       120Gi
Swap:          2.0Gi          0B       2.0Gi

--- 3. 120TB+ 磁盘阵列检查 ---
sda           8:0    0   7.3T  0 disk 
sdb           8:16   0   9.1T  0 disk 
sdc           8:32   0   9.1T  0 disk 
sdd           8:48   0   9.1T  0 disk 
sde           8:64   0   9.1T  0 disk 
sdf           8:80   0   9.1T  0 disk 
sdg           8:96   0   9.1T  0 disk 
sdh 

几个进阶小技巧：
将结果存入 Python 变量：
如果你想在 Python 里处理这些数据，可以这样做：

In [4]:
mem_info = !free -h
print(f"我的服务器内存现状是: {mem_info[1]}")

我的服务器内存现状是: Mem:           125Gi       4.8Gi       115Gi       9.0Mi       5.7Gi       119Gi


2. 关于 sudo：
如果在 Notebook 里运行需要权限的命令（如 sudo dmidecode），通常会报错因为无法交互输入密码。如果必须运行，可以尝试：
!echo '你的密码' | sudo -S command （注意： 这会在代码里暴露密码，仅限私人安全环境使用）。

3. 不要运行交互式命令：
在 Notebook 里千万不要运行像 top, htop, glances 或 watch 这种需要持续刷新的命令，它们会导致 Jupyter 单元格永远卡在运行状态。请使用上面提供的静态显示命令。

在 Jupyter Notebook 中监控系统盘和外接盘非常重要。/dev/nvme0n1p2 是你的 NVMe 固态硬盘（高速总线直连 CPU），它的命名规则与 USB/SATA 硬盘完全不同。
1. 更新后的 Jupyter 检查代码 (包含 NVMe)
你可以运行这个单元格来查看所有硬盘（包括你的系统盘、12 块 10T 机械硬盘和 8T 外接 SSD）：

In [5]:
%%bash
echo "=== 1. 系统盘 (NVMe) 与 所有硬盘挂载状态 ==="
# -e 7 排除回路设备，grep 显示 NVMe 和所有 sdX 硬盘
df -h | grep -E "Filesystem|/dev/nvme|/dev/sd"

echo -e "\n=== 2. 物理硬盘列表 (设备层次结构) ==="
# lsblk 能够清晰显示 NVMe (系统盘) 和 sdX (外接盘)
lsblk -o NAME,SIZE,TYPE,MOUNTPOINT,MODEL | grep -E "NAME|nvme|sd"

=== 1. 系统盘 (NVMe) 与 所有硬盘挂载状态 ===
Filesystem      Size  Used Avail Use% Mounted on
/dev/nvme0n1p2  916G  130G  739G  15% /
/dev/nvme0n1p1  511M  6.1M  505M   2% /boot/efi
/dev/sda1       7.3T  4.1G  7.3T   1% /mnt/ex_8T_SSD
/dev/sdc1       9.1T   16K  8.6T   1% /srv/external_data
/dev/sdb1       9.1T  7.0T  1.7T  81% /Work_bio

=== 2. 物理硬盘列表 (设备层次结构) ===
NAME          SIZE TYPE MOUNTPOINT                          MODEL
sda           7.3T disk                                     Desk SSD
└─sda1        7.3T part /mnt/ex_8T_SSD                      
sdb           9.1T disk                                     ST10000VN0008-2JJ101
└─sdb1        9.1T part /Work_bio                           
sdc           9.1T disk                                     ST10000VN0008-2JJ101
└─sdc1        9.1T part /srv/external_data                  
sdd           9.1T disk                                     ST10000VN0004-1ZD101
└─sdd1        9.1T part                                     
sde           9.1T dis

In [ ]:
2. 硬盘命名规律详解
Linux 下的硬盘命名不是随意的，它反映了物理接口和连接顺序：


In [ ]:
(1) NVMe 固态硬盘 (你的系统盘)
命名格式：/dev/nvme[控制器号]n[命名空间号]p[分区号]
nvme0: 表示第 0 个 NVMe 控制器（主板上第一个 M.2 槽位）。
n1: 表示第一个 Namespace（通常一个物理盘就是一个 n1）。
p2: 表示 Partition 2（第 2 个分区）。通常 p1 是 EFI 引导分区，p2 是你的 Ubuntu 系统根目录 /。
优点：命名非常固定，通常不会因为插拔其他 USB 硬盘而改变。


In [ ]:
(2) SATA/USB 硬盘 (你的 12 块 10T 盘和 8T 外接 SSD)
命名格式：/dev/sd[字母][数字]
sd: 代表 "SCSI Disk"（现在所有的 SATA 硬盘和 USB 硬盘都沿用这个前缀）。
a, b, c...: 代表系统识别到的先后顺序。
/dev/sda 是第一个被识别到的。
/dev/sdb 是第二个，以此类推。
风险点：如果你重启服务器时插拔了 USB 线，原本是 sda 的硬盘下次可能会变成 sdb。这就是为什么在 /etc/fstab 里通常用 UUID 而不是路径来挂载。
数字: 代表分区。例如 /dev/sda1 是第一块硬盘的第一个分区。


In [ ]:
(3) 为什么你的 8T 杀手 SSD 跑出 175MB/s？
蓝口 (USB 3.0)：带宽限制了它的发挥。
命名：它在 lsblk 中应该也会显示为 /dev/sdX（某个字母）。
规律：在你的机器上，由于 12 块机械硬盘数量庞大，你的 8T SSD 可能会排在比较靠后的字母（比如 /dev/sdm 或 /dev/sdn）。

3. 给管理员的“职业级”建议
别被 /dev/nvme0n1 吓到：这是目前最先进的驱动协议，它的性能是你那些外接机械硬盘的 10-20 倍。
监控 IO 压力：
由于你挂载了这么多大容量硬盘，如果想看哪块盘正在忙，可以运行： (如果没安装：!sudo apt install sysstat)

In [6]:
!iostat -xz 1 2

Linux 6.8.0-111-generic (us1) 	05/28/2026 	_x86_64_	(64 CPU)

avg-cpu:  %user   %nice %system %iowait  %steal   %idle
           0.76    0.00    0.39    0.12    0.00   98.73

Device            r/s     rkB/s   rrqm/s  %rrqm r_await rareq-sz     w/s     wkB/s   wrqm/s  %wrqm w_await wareq-sz     d/s     dkB/s   drqm/s  %drqm d_await dareq-sz     f/s f_await  aqu-sz  %util
loop0            0.00      0.00     0.00   0.00    0.00     1.21    0.00      0.00     0.00   0.00    0.00     0.00    0.00      0.00     0.00   0.00    0.00     0.00    0.00    0.00    0.00   0.00
loop1            0.04      0.12     0.00   0.00    0.06     2.94    0.00      0.00     0.00   0.00    0.00     0.00    0.00      0.00     0.00   0.00    0.00     0.00    0.00    0.00    0.00   0.00
loop10           0.01      0.14     0.00   0.00    1.12    17.27    0.00      0.00     0.00   0.00    0.00     0.00    0.00      0.00     0.00   0.00    0.00     0.00    0.00    0.00    0.00   0.00
loop11           0.01      0.14  

确认系统盘健康： DA
既然昨天经历了断电，建议看一眼 NVMe 的健康状态：

In [7]:
!sudo smartctl -a /dev/nvme0n1 | grep "percentage_used"

[sudo] password for gao: 


In [8]:
!cat /etc/fstab #是查看服务器“硬盘自动挂载配置”的最高效方式。

# /etc/fstab: static file system information.
#
# Use 'blkid' to print the universally unique identifier for a
# device; this may be used with UUID= as a more robust way to name devices
# that works even if disks are added and removed. See fstab(5).
#
# <file system> <mount point>   <type>  <options>       <dump>  <pass>
# / was on /dev/nvme0n1p2 during installation
UUID=7d38d158-8f0c-404a-8603-ed7d25cae5a3 /               ext4    errors=remount-ro 0       1
# /boot/efi was on /dev/nvme0n1p1 during installation
UUID=8114-C92A  /boot/efi       vfat    umask=0077      0       1
/swapfile                                 none            swap    sw              0       0
UUID=869c4c24-370e-4708-9bf0-cccbda693df9  /Work_bio  ext4  defaults,nofail 0  2
UUID=1166d376-fc2e-45ff-86e8-c3b35b0fae51 /srv/external_data  ext4  defaults,nofail 0  2
UUID=2623-0697 /mnt/ex_8T_SSD exfat defaults,nofail,uid=1000,gid=1000,umask=000 0 0
